In [ ]:
%load_ext autoreload
%autoreload 2
# Set up GPU rendering.
# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl
import mujoco

# Other imports and helper functions
import numpy as np

from mujoco_irb120.util.load_obj_in_env import load_environment, load_photoshoot
import mujoco_irb120.controllers.robot_controller as robot_controller
from mujoco_irb120.util.helper_fns import *
from mujoco_irb120.util.render_opts import RendererViewerOpts

# Graphics and plotting.
import mediapy as media
import matplotlib.pyplot as plt
import json as _json

%matplotlib ipympl

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)
# Set matplotlib font size
fonts = {'size' : 20}
plt.rc('font', **fonts)
# %matplotlib notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Setting environment variable to use GPU rendering:
env: MUJOCO_GL=egl


#### (Optional) Keyboard Cartesian Control Setup
To enable interactive keyboard control of the robot end-effector:
- **Arrow Keys**: Move in X and Z directions
  - LEFT: Move -X (away from object)
  - RIGHT: Move +X (toward object)
  - UP: Move +Z (lift)
  - DOWN: Move -Z (lower)

Set `KEYBOARD_CONTROL = True` below to enable. Requires `VIZ = True` and focus on the viewer window.

In [43]:
# Enable/disable keyboard control toggle
KEYBOARD_CONTROL = True   # Set to False to disable keyboard control
print(f"Keyboard control: {'ENABLED' if KEYBOARD_CONTROL else 'DISABLED'}")
if KEYBOARD_CONTROL:
    print("  Controls: Arrow keys to move in X and Z directions")
    print("  Make sure viewer window has focus for keyboard input")

Keyboard control: ENABLED
  Controls: Arrow keys to move in X and Z directions
  Make sure viewer window has focus for keyboard input


## Main Simulation Loop

In [ ]:
# ======================== Toggle visualization here =========================
VIZ = True   # set to False to record video without showing the viewer
# ============================================================================

## Let's recall the model to reset the simulation
# 0: box_exp, 10: heart, 11: L_shape, 12: monitor, 13: soda, 14: flashlight
OBJECT = 0

model, data = load_environment(num=OBJECT, launch_viewer=False)

## =================== LOAD GROUND TRUTH PARAMS FROM JSON ===================
_obj_params = _json.load(open("object_params.json"))["objects"][str(OBJECT)]

com_GT   = np.subtract(_obj_params["com_gt_onshape"], _obj_params["com_gt_offset"])
mass_GT     = _obj_params["mass_gt"]
init_xyz = np.array(_obj_params["init_xyz"])
## ===================================================================

## Setup based on robot model
irb = robot_controller.controller(model, data)

# Get initial EE pose (finger tip)
T_home = irb.FK()
print('Initial end-effector pose:\n', T_home)

## Set robot just in front of payload (same orientation as home position (facing +x))
T_init = T_home.copy()
T_init[:3, 3] = init_xyz.copy()

q_init = irb.IK(T_init, method=2, damping=0.5, max_iters=1000) # DLS method
irb.set_pose(q=q_init)

## The end pose we want to reach FOR POSITION CONTROL (format: 4x4 matrix)
T_end = T_init.copy()
T_end[0, 3] += 0.10  # Move EE forward by 15 cm in x direction

target_q = irb.IK(T_end, method=2, damping=0.5, max_iters=1000)  # DLS method

## TARE / Bias sensor
irb.ft_bias(n_samples=200)

## FOR VELOCITY CONTROL (format: [wx wy wz vx vy vz])
# target_vel  = np.array([0.0, 0.0, 0.0, 0.14, 0.0, 0.0])  # Move EE forward at 4 cm/s in x direction

## Initialize time, force and tilt history for plotting
t_hist          = []
w_hist          = []
quat_hist       = []
ball_pose_hist  = []  # (4,4) pose of ball-center site in world frame
sens_pose_hist  = []  # (4,4) pose of FT sensor site in world frame
con_bool_hist   = []  # contact flag
obj_pose_hist   = []  # (4,4) object pose in world frame (mj internal)

traj_duration = 6.0 # seconds
run_duration = traj_duration + 50.0 # 4.0  # seconds

# Initialize trajectory recorder for recording or load trajectory for playback
recorder = None
loaded_trajectory = None

if MOTION_MODE == 'RECORD':
    recorder = TrajectoryRecorder(irb)
    recorder.start_recording(verbose=False, record_forces=RECORD_FORCES)
elif MOTION_MODE == 'PLAYBACK':
    recorder = TrajectoryRecorder(irb)
    try:
        loaded_trajectory = recorder.load_trajectory("recorded_motion.npz")
        recorder.start_playback(trajectory=loaded_trajectory)
        print("Loaded and ready for playback.")
    except FileNotFoundError:
        print("ERROR: recorded_motion.npz not found. Skipping playback.")
        MOTION_MODE = None

## Additions for video recording
rv = RendererViewerOpts(model, data, vis=VIZ, show_left_UI=True)
# ===========================================================================
with rv: # enters viewer if vis=True, sets viewer opts, and readies offscreen renderer for video capture
    while rv.viewer_is_running() and not irb.stop and data.time < run_duration:
        irb.check_topple()                          # Check for payload topple condition

        # Apply control: either keyboard-based or trajectory-based position control
        if KEYBOARD_CONTROL and VIZ:
            v_cmd = rv.get_keyboard_input()
            irb.apply_cartesian_keyboard_ctrl(v_cmd, maintain_orientation=True, verbose=False)
        else:
            if data.time < traj_duration:
                alpha = data.time / traj_duration
                interp_q = (1 - alpha) * q_init + alpha * target_q
            else:
                interp_q = target_q.copy()
            irb.set_pos_ctrl(interp_q, check_ellipsoid=False)

        # Record or playback waypoint if enabled
        if MOTION_MODE == 'RECORD' and recorder:
            recorder.record_waypoint(record_type='joints')
        elif MOTION_MODE == 'PLAYBACK' and recorder:
            recorder.playback_step(playback_type='joints', interpolate=True)

        mujoco.mj_step(model, data)                 # Step the simulation

        w_hist.append(irb.ft_get_reading())
        quat_hist.append(irb.get_payload_pose(out='quat'))
        t_hist.append(data.time)
        ball_pose_hist.append(irb.get_site_pose("ball"))
        sens_pose_hist.append(irb.get_site_pose("sensor"))
        con_bool_hist.append(irb.check_contact())
        obj_pose_hist.append(irb.get_payload_pose(out='T'))

        rv.sync()
        rv.capture_frame_if_due(data)

t_hist          = np.asarray(t_hist,         dtype=float)
quat_hist       = np.asarray(quat_hist,      dtype=float)
con_bool_hist   = np.asarray(con_bool_hist,  dtype=float)
w_hist          = np.asarray(w_hist,         dtype=float).reshape(-1, 6)
ball_pose_hist  = np.asarray(ball_pose_hist, dtype=float).reshape(-1, 4, 4)
sens_pose_hist  = np.asarray(sens_pose_hist, dtype=float).reshape(-1, 4, 4)
obj_pose_hist   = np.asarray(obj_pose_hist,  dtype=float).reshape(-1, 4, 4)

ball_pos_hist = ball_pose_hist[:, :3, 3]
sens_pos_hist = sens_pose_hist[:, :3, 3]
obj_pos_hist  = obj_pose_hist[:,  :3, 3]

# Save trajectory if recording was enabled (skip if playback mode)
if MOTION_MODE == 'RECORD' and recorder:
    recorder.stop_recording(verbose=False)
    recorder.save_trajectory("recorded_motion.npz", format='numpy')
elif MOTION_MODE == 'PLAYBACK' and recorder:
    recorder.stop_playback()

print(f'\nSimulation ended in t = {data.time:.2f} seconds.')
media.show_video(rv.frames, fps=rv.framerate)

Initial end-effector pose:
 [[1.    0.    0.    0.374]
 [0.    1.    0.    0.   ]
 [0.    0.    1.    0.665]
 [0.    0.    0.    1.   ]]
Biasing F/T sensor with 200 static samples...
Force offset: [-0.    -1.256  0.     0.089 -0.    -0.   ]

Simulation ended in t = 10.39 seconds.


This browser does not support the video tag.


In [45]:
# Friction sanity check (tangent coefficient between object and table).
table_geom_id = model.geom('table').id
mu_table_tangent = float(model.geom_friction[table_geom_id, 0])

# Use explicit payload geom name when present; fallback is last geom.
try:
    payload_geom_id = model.geom('payload').id
except Exception:
    payload_geom_id = model.ngeom - 1

payload_geom_name = model.geom(payload_geom_id).name
mu_obj_tangent = float(model.geom_friction[payload_geom_id, 0])

# MuJoCo uses the higher-priority geom if present; otherwise it takes the element-wise max.
mu_obj_table_tangent = float(max(mu_table_tangent, mu_obj_tangent))

In [46]:
# # ====== OPTIONAL: Save all simulation variables to numpy file ======
# # This captures all the data collected during the simulation loop
# # Useful when you want to preserve force/pose history even if not using trajectory recorder

np.savez(
    "simulation_data.npz",
    t_hist=t_hist,
    w_hist=w_hist,                    # Force/torque at each step (N, Nm)
    quat_hist=quat_hist,              # Object quaternion at each step
    ball_pose_hist=ball_pose_hist,    # Ball-center pose (4x4 transforms)
    sens_pose_hist=sens_pose_hist,    # FT sensor pose (4x4 transforms)
    con_bool_hist=con_bool_hist,      # Contact status at each step
    obj_pose_hist=obj_pose_hist,      # Object pose (4x4 transforms)
    ball_pos_hist=ball_pos_hist,      # Ball position trajectory
    sens_pos_hist=sens_pos_hist,      # Sensor position trajectory
    obj_pos_hist=obj_pos_hist,        # Object position trajectory
    com_gt=com_GT,                    # Ground truth center of mass (3,)
    mass_gt=mass_GT,                  # Ground truth mass
    mu_gt=mu_obj_table_tangent        # Effective tangent friction coefficient between object and table
)
print("Saved all simulation data to simulation_data.npz")

Saved all simulation data to simulation_data.npz
